# PointNeXt ModelNet40 free-GPU smoke/eval

This notebook targets Google Colab Free or Kaggle free GPU. It installs PointNeXt/OpenPoints from source, builds the CUDA ops, downloads the ModelNet40 PointNeXt-S C=64 checkpoint from Hugging Face Hub, verifies SHA-256 when the manifest is available, and runs the ModelNet40 test command.

A T4/P100 is enough for this smoke path. Full S3DIS/ScanNet training is not a free-GPU target.


In [ ]:
import torch, platform
print('python:', platform.python_version())
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Enable GPU runtime: Runtime -> Change runtime type -> GPU'


In [ ]:
# Clone with submodules. If you are testing a PR branch, replace --branch master.
!git clone --recurse-submodules https://github.com/guochengqian/PointNeXt.git
%cd PointNeXt
!git submodule update --init --recursive


In [ ]:
# Install Python deps. Colab's preinstalled PyTorch is usually OK; avoid reinstalling torch unless needed.
!python -m pip install -U pip setuptools wheel ninja gdown huggingface_hub
!python -m pip install -r requirements.txt --no-deps
!python -m pip install -e openpoints
!python -m pip install -e .


In [ ]:
# Build the CUDA ops used by PointNeXt. This takes a few minutes on Colab Free.
!cd openpoints/cpp/pointnet2_batch && python setup.py install
!cd openpoints/cpp/pointops && python setup.py install
# chamfer/emd are optional for classification, but building them makes the environment closer to a full source install.
!cd openpoints/cpp/chamfer_dist && python setup.py install
!cd openpoints/cpp/emd && python setup.py install


In [ ]:
# Download the ModelNet40 PointNeXt-S C=64 checkpoint.
# If the HF repo is still private, set HF_TOKEN in Colab secrets and pass --token.
!python tools/download_checkpoint.py modelnet40-pointnext-s-c64 --output-dir ./hf_cache


In [ ]:
# Run ModelNet40 evaluation. The dataset is downloaded automatically by OpenPoints.
CKPT='hf_cache/checkpoints/modelnet40/pointnext-s-c64.pth'
!CUDA_VISIBLE_DEVICES=0 python examples/classification/main.py --cfg cfgs/modelnet40ply2048/pointnext-s.yaml model.encoder_args.width=64 mode=test --pretrained_path $CKPT wandb.use_wandb=False dataloader.num_workers=2


Expected released checkpoint result: about OA 94.0 / mAcc 91.1 on ModelNet40. For a quick smoke check, successful CUDA op import, checkpoint download/hash verification, dataset load, and a completed test loop are the main pass criteria.
